# Advanced RAG con Capacidades de Agente

Este notebook implementa un sistema RAG avanzado que combina:
- 🔍 Recuperación de información de documentos
- 🧮 Capacidades de cálculo y programación
- 📊 Generación de visualizaciones
- 📈 Análisis estadístico

El LLM puede entender archivos, realizar cálculos complejos y generar visualizaciones basadas en preguntas en lenguaje natural.

## 1. Instalación de Dependencias

In [ ]:
!pip install -q anthropic chromadb pandas matplotlib seaborn numpy scipy scikit-learn plotly

## 2. Imports y Configuración

In [ ]:
import anthropic
import chromadb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from sklearn.linear_model import LinearRegression
import json
import os
from typing import List, Dict, Any
import re
from pathlib import Path

# Configuración de estilo para visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# API Key de Anthropic
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "tu-api-key-aqui")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

## 3. Sistema RAG - Gestión de Documentos

In [ ]:
class DocumentRAG:
    """Sistema RAG para gestionar documentos y realizar búsquedas semánticas"""
    
    def __init__(self, collection_name="advanced_rag"):
        self.chroma_client = chromadb.Client()
        self.collection = self.chroma_client.create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        self.documents = []
    
    def add_document(self, text: str, metadata: dict = None):
        """Agrega un documento a la base de datos vectorial"""
        doc_id = f"doc_{len(self.documents)}"
        self.documents.append({"id": doc_id, "text": text, "metadata": metadata or {}})
        
        self.collection.add(
            documents=[text],
            ids=[doc_id],
            metadatas=[metadata or {}]
        )
        return doc_id
    
    def add_file(self, file_path: str):
        """Agrega un archivo de texto al RAG"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                content = f.read()
            
            metadata = {
                "filename": os.path.basename(file_path),
                "filepath": file_path,
                "type": "file"
            }
            return self.add_document(content, metadata)
        except Exception as e:
            print(f"Error leyendo archivo {file_path}: {e}")
            return None
    
    def search(self, query: str, n_results: int = 3) -> List[str]:
        """Busca documentos relevantes basados en la consulta"""
        if len(self.documents) == 0:
            return []
        
        results = self.collection.query(
            query_texts=[query],
            n_results=min(n_results, len(self.documents))
        )
        
        return results['documents'][0] if results['documents'] else []

# Inicializar RAG
rag = DocumentRAG()
print("✅ Sistema RAG inicializado")

## 4. Herramientas para el Agente

Definimos las herramientas que el LLM puede usar mediante function calling.

In [ ]:
# Definición de herramientas disponibles para el agente
TOOLS = [
    {
        "name": "calculator",
        "description": "Ejecuta cálculos matemáticos complejos. Puede evaluar expresiones matemáticas, funciones científicas, estadísticas, etc. Usa Python eval() con acceso a numpy, scipy.stats.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Expresión matemática a evaluar en Python. Ejemplo: 'np.sqrt(16) + np.mean([1,2,3,4,5])'"
                }
            },
            "required": ["expression"]
        }
    },
    {
        "name": "execute_python",
        "description": "Ejecuta código Python arbitrario para análisis de datos, estadísticas, transformaciones, etc. Tiene acceso a pandas, numpy, scipy, sklearn.",
        "input_schema": {
            "type": "object",
            "properties": {
                "code": {
                    "type": "string",
                    "description": "Código Python a ejecutar. Debe asignar el resultado a una variable 'result'."
                }
            },
            "required": ["code"]
        }
    },
    {
        "name": "create_visualization",
        "description": "Crea visualizaciones de datos (gráficos, plots, charts). Soporta múltiples tipos: line, bar, scatter, histogram, box, heatmap, pie.",
        "input_schema": {
            "type": "object",
            "properties": {
                "chart_type": {
                    "type": "string",
                    "enum": ["line", "bar", "scatter", "histogram", "box", "heatmap", "pie"],
                    "description": "Tipo de gráfico a crear"
                },
                "data": {
                    "type": "object",
                    "description": "Datos para el gráfico en formato dict. Ej: {'x': [1,2,3], 'y': [4,5,6]}"
                },
                "title": {
                    "type": "string",
                    "description": "Título del gráfico"
                },
                "labels": {
                    "type": "object",
                    "description": "Etiquetas de ejes. Ej: {'x': 'Tiempo', 'y': 'Valor'}"
                }
            },
            "required": ["chart_type", "data"]
        }
    },
    {
        "name": "statistical_analysis",
        "description": "Realiza análisis estadístico de datos: media, mediana, desviación estándar, correlación, regresión, tests estadísticos, etc.",
        "input_schema": {
            "type": "object",
            "properties": {
                "analysis_type": {
                    "type": "string",
                    "enum": ["descriptive", "correlation", "regression", "t_test", "anova"],
                    "description": "Tipo de análisis estadístico"
                },
                "data": {
                    "type": "object",
                    "description": "Datos a analizar en formato dict o list"
                }
            },
            "required": ["analysis_type", "data"]
        }
    },
    {
        "name": "search_documents",
        "description": "Busca información relevante en los documentos cargados en el sistema RAG.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Consulta de búsqueda"
                },
                "n_results": {
                    "type": "integer",
                    "description": "Número de resultados a retornar (default: 3)"
                }
            },
            "required": ["query"]
        }
    }
]

print("✅ Herramientas definidas:")
for tool in TOOLS:
    print(f"  - {tool['name']}: {tool['description'][:60]}...")

## 5. Implementación de Herramientas

In [ ]:
def calculator(expression: str) -> str:
    """Evalúa expresiones matemáticas"""
    try:
        # Crear namespace seguro con funciones matemáticas
        safe_dict = {
            'np': np,
            'stats': stats,
            'sqrt': np.sqrt,
            'sin': np.sin,
            'cos': np.cos,
            'tan': np.tan,
            'log': np.log,
            'exp': np.exp,
            'mean': np.mean,
            'median': np.median,
            'std': np.std,
            'sum': np.sum,
            'pi': np.pi,
            'e': np.e
        }
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"Resultado: {result}"
    except Exception as e:
        return f"Error en cálculo: {str(e)}"

def execute_python(code: str) -> str:
    """Ejecuta código Python"""
    try:
        local_vars = {
            'pd': pd,
            'np': np,
            'stats': stats,
            'result': None
        }
        exec(code, {"__builtins__": __builtins__}, local_vars)
        return f"Resultado: {local_vars.get('result', 'No result variable set')}"
    except Exception as e:
        return f"Error ejecutando código: {str(e)}"

def create_visualization(chart_type: str, data: dict, title: str = "", labels: dict = None) -> str:
    """Crea visualizaciones de datos"""
    try:
        plt.figure(figsize=(10, 6))
        
        if chart_type == "line":
            plt.plot(data.get('x', range(len(data['y']))), data['y'])
        elif chart_type == "bar":
            plt.bar(data.get('x', range(len(data['y']))), data['y'])
        elif chart_type == "scatter":
            plt.scatter(data['x'], data['y'])
        elif chart_type == "histogram":
            plt.hist(data['values'], bins=data.get('bins', 10))
        elif chart_type == "box":
            plt.boxplot(data['values'])
        elif chart_type == "heatmap":
            sns.heatmap(data['matrix'], annot=True, cmap='coolwarm')
        elif chart_type == "pie":
            plt.pie(data['values'], labels=data.get('labels'), autopct='%1.1f%%')
        
        if title:
            plt.title(title)
        if labels:
            plt.xlabel(labels.get('x', ''))
            plt.ylabel(labels.get('y', ''))
        
        plt.tight_layout()
        plt.show()
        return f"✅ Gráfico '{chart_type}' creado exitosamente"
    except Exception as e:
        return f"Error creando visualización: {str(e)}"

def statistical_analysis(analysis_type: str, data: dict) -> str:
    """Realiza análisis estadístico"""
    try:
        if analysis_type == "descriptive":
            values = np.array(data['values'])
            result = {
                'mean': np.mean(values),
                'median': np.median(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values),
                'q25': np.percentile(values, 25),
                'q75': np.percentile(values, 75)
            }
            return json.dumps(result, indent=2)
        
        elif analysis_type == "correlation":
            x = np.array(data['x'])
            y = np.array(data['y'])
            correlation = np.corrcoef(x, y)[0, 1]
            return f"Correlación: {correlation:.4f}"
        
        elif analysis_type == "regression":
            x = np.array(data['x']).reshape(-1, 1)
            y = np.array(data['y'])
            model = LinearRegression()
            model.fit(x, y)
            r2 = model.score(x, y)
            return f"Regresión lineal - Slope: {model.coef_[0]:.4f}, Intercept: {model.intercept_:.4f}, R²: {r2:.4f}"
        
        elif analysis_type == "t_test":
            group1 = np.array(data['group1'])
            group2 = np.array(data['group2'])
            t_stat, p_value = stats.ttest_ind(group1, group2)
            return f"T-test - t-statistic: {t_stat:.4f}, p-value: {p_value:.4f}"
        
        return "Tipo de análisis no soportado"
    except Exception as e:
        return f"Error en análisis estadístico: {str(e)}"

def search_documents(query: str, n_results: int = 3) -> str:
    """Busca en documentos del RAG"""
    results = rag.search(query, n_results)
    if not results:
        return "No se encontraron documentos relevantes"
    return "\n\n---\n\n".join(results)

# Mapeo de funciones
TOOL_FUNCTIONS = {
    "calculator": calculator,
    "execute_python": execute_python,
    "create_visualization": create_visualization,
    "statistical_analysis": statistical_analysis,
    "search_documents": search_documents
}

print("✅ Implementaciones de herramientas cargadas")

## 6. Agente con Function Calling

In [ ]:
class AdvancedRAGAgent:
    """Agente RAG avanzado con capacidades de cálculo y visualización"""
    
    def __init__(self, model="claude-3-5-sonnet-20241022", max_iterations=10):
        self.model = model
        self.max_iterations = max_iterations
        self.conversation_history = []
    
    def process_tool_call(self, tool_name: str, tool_input: dict) -> str:
        """Procesa una llamada a herramienta"""
        print(f"\n🔧 Ejecutando herramienta: {tool_name}")
        print(f"   Parámetros: {json.dumps(tool_input, indent=2)}")
        
        if tool_name in TOOL_FUNCTIONS:
            result = TOOL_FUNCTIONS[tool_name](**tool_input)
            print(f"   ✅ Resultado: {result[:100]}..." if len(str(result)) > 100 else f"   ✅ Resultado: {result}")
            return result
        else:
            return f"Error: herramienta '{tool_name}' no encontrada"
    
    def chat(self, user_message: str, context: str = None) -> str:
        """Procesa un mensaje del usuario con capacidades de agente"""
        print(f"\n{'='*80}")
        print(f"👤 Usuario: {user_message}")
        print(f"{'='*80}")
        
        # Construir mensaje del sistema
        system_message = """Eres un asistente avanzado con capacidades de análisis de datos, cálculo y visualización.
        
Puedes:
- Realizar cálculos matemáticos y estadísticos complejos
- Ejecutar código Python para análisis de datos
- Crear visualizaciones (gráficos, plots, charts)
- Buscar información en documentos
- Realizar análisis estadísticos avanzados

Cuando el usuario haga una pregunta:
1. Analiza qué herramientas necesitas usar
2. Usa las herramientas apropiadas
3. Presenta los resultados de forma clara y concisa

Siempre responde en español."""
        
        if context:
            system_message += f"\n\nContexto adicional de documentos:\n{context}"
        
        # Agregar mensaje a historial
        messages = self.conversation_history + [{"role": "user", "content": user_message}]
        
        # Loop de agente
        for iteration in range(self.max_iterations):
            response = client.messages.create(
                model=self.model,
                max_tokens=4096,
                system=system_message,
                messages=messages,
                tools=TOOLS
            )
            
            # Procesar respuesta
            if response.stop_reason == "end_turn":
                # Respuesta final
                final_response = ""
                for block in response.content:
                    if hasattr(block, "text"):
                        final_response += block.text
                
                print(f"\n🤖 Asistente: {final_response}")
                
                # Actualizar historial
                self.conversation_history.append({"role": "user", "content": user_message})
                self.conversation_history.append({"role": "assistant", "content": response.content})
                
                return final_response
            
            elif response.stop_reason == "tool_use":
                # Procesar tool uses
                tool_results = []
                
                for block in response.content:
                    if block.type == "tool_use":
                        tool_result = self.process_tool_call(block.name, block.input)
                        tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(tool_result)
                        })
                
                # Continuar conversación con resultados
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": tool_results})
            
            else:
                print(f"\n⚠️  Stop reason inesperado: {response.stop_reason}")
                break
        
        return "Se alcanzó el límite de iteraciones"
    
    def reset_conversation(self):
        """Reinicia el historial de conversación"""
        self.conversation_history = []
        print("✅ Historial de conversación reiniciado")

# Inicializar agente
agent = AdvancedRAGAgent()
print("\n✅ Agente RAG avanzado inicializado y listo")

## 7. Cargar Documentos de Ejemplo

In [ ]:
# Ejemplo: Agregar datos sobre ventas
ventas_doc = """
REPORTE DE VENTAS - Q1 2024

Ventas mensuales:
- Enero: $45,000 (150 unidades)
- Febrero: $52,000 (173 unidades)
- Marzo: $61,000 (203 unidades)

Total Q1: $158,000
Crecimiento: 35% respecto a Q1 2023

Productos más vendidos:
1. Producto A - 180 unidades
2. Producto B - 145 unidades
3. Producto C - 101 unidades

Regiones:
- Norte: 40% de ventas
- Sur: 35% de ventas
- Este: 25% de ventas
"""

rag.add_document(ventas_doc, {"type": "sales_report", "period": "Q1_2024"})

# Agregar más documentos de ejemplo
tech_doc = """
ESPECIFICACIONES TÉCNICAS - Sistema Analytics

El sistema procesa aproximadamente 10,000 eventos por segundo.
Latencia promedio: 45ms
Uptime: 99.9%

Stack tecnológico:
- Backend: Python 3.11, FastAPI
- Base de datos: PostgreSQL 15
- Cache: Redis 7.0
- ML Models: TensorFlow 2.15

Métricas de rendimiento:
- CPU usage promedio: 35%
- Memoria: 8GB/16GB utilizados
- Disco: 120GB/500GB
"""

rag.add_document(tech_doc, {"type": "technical_specs", "system": "analytics"})

print("✅ Documentos de ejemplo cargados en el RAG")
print(f"   Total de documentos: {len(rag.documents)}")

## 8. Ejemplos de Uso

### Ejemplo 1: Cálculos Matemáticos

In [ ]:
response = agent.chat(
    "Calcula la raíz cuadrada de 144 y luego multiplícala por el promedio de estos números: 10, 20, 30, 40, 50"
)

### Ejemplo 2: Análisis Estadístico con Datos del RAG

In [ ]:
response = agent.chat(
    "Busca los datos de ventas mensuales en los documentos y calcula las estadísticas descriptivas (media, mediana, desviación estándar)"
)

### Ejemplo 3: Crear Visualización

In [ ]:
response = agent.chat(
    "Crea un gráfico de barras que muestre las ventas mensuales de Enero, Febrero y Marzo con valores de 45000, 52000 y 61000 respectivamente. Título: 'Ventas Q1 2024'"
)

### Ejemplo 4: Análisis Complejo con Múltiples Herramientas

In [ ]:
response = agent.chat(
    """Quiero un análisis completo de las ventas:
    1. Busca los datos de ventas en los documentos
    2. Calcula el crecimiento porcentual entre cada mes
    3. Crea una visualización de línea mostrando la tendencia
    4. Dame un análisis estadístico descriptivo
    """
)

### Ejemplo 5: Ejecutar Código Python Personalizado

In [ ]:
response = agent.chat(
    """Ejecuta un código Python que:
    - Cree un array de 100 números aleatorios entre 0 y 100
    - Calcule cuántos son mayores a 50
    - Calcule el promedio de los que son mayores a 50
    Asigna el resultado como un diccionario a la variable 'result'
    """
)

### Ejemplo 6: Análisis de Correlación y Regresión

In [ ]:
response = agent.chat(
    """Tengo estos datos:
    - Inversión en marketing (miles): [10, 15, 20, 25, 30, 35]
    - Ventas (miles): [45, 52, 61, 68, 75, 82]
    
    Analiza la correlación entre inversión y ventas, y crea un modelo de regresión lineal.
    Luego crea un gráfico de dispersión mostrando los puntos y la línea de tendencia.
    """
)

## 9. Funciones Auxiliares para Uso Rápido

In [ ]:
def quick_calc(expression: str):
    """Función rápida para cálculos"""
    return agent.chat(f"Calcula: {expression}")

def quick_viz(data: dict, chart_type: str = "line", title: str = ""):
    """Función rápida para visualizaciones"""
    prompt = f"Crea un gráfico tipo {chart_type} con estos datos: {data}"
    if title:
        prompt += f" con título '{title}'"
    return agent.chat(prompt)

def quick_stats(data: list):
    """Función rápida para estadísticas descriptivas"""
    return agent.chat(f"Calcula estadísticas descriptivas de estos datos: {data}")

def ask_documents(question: str):
    """Función rápida para buscar en documentos"""
    return agent.chat(f"Busca en los documentos: {question}")

print("✅ Funciones auxiliares cargadas:")
print("   - quick_calc(expression): Cálculos rápidos")
print("   - quick_viz(data, chart_type, title): Visualizaciones rápidas")
print("   - quick_stats(data): Estadísticas descriptivas")
print("   - ask_documents(question): Búsqueda en documentos")

## 10. Modo Interactivo

In [ ]:
def interactive_mode():
    """Modo interactivo para chatear con el agente"""
    print("\n" + "="*80)
    print("MODO INTERACTIVO - Advanced RAG Agent")
    print("="*80)
    print("Comandos especiales:")
    print("  /reset - Reiniciar conversación")
    print("  /docs - Ver documentos cargados")
    print("  /add - Agregar documento")
    print("  /exit - Salir")
    print("="*80 + "\n")
    
    while True:
        try:
            user_input = input("\n👤 Tú: ").strip()
            
            if not user_input:
                continue
            
            if user_input == "/exit":
                print("\n👋 ¡Hasta luego!")
                break
            
            elif user_input == "/reset":
                agent.reset_conversation()
                continue
            
            elif user_input == "/docs":
                print(f"\n📚 Documentos cargados: {len(rag.documents)}")
                for i, doc in enumerate(rag.documents):
                    print(f"  {i+1}. {doc['metadata']}")
                continue
            
            elif user_input == "/add":
                print("\nEscribe el texto del documento (termina con línea vacía):")
                lines = []
                while True:
                    line = input()
                    if not line:
                        break
                    lines.append(line)
                doc_text = "\n".join(lines)
                doc_id = rag.add_document(doc_text)
                print(f"✅ Documento agregado con ID: {doc_id}")
                continue
            
            # Procesar mensaje normal
            agent.chat(user_input)
            
        except KeyboardInterrupt:
            print("\n\n👋 ¡Hasta luego!")
            break
        except Exception as e:
            print(f"\n❌ Error: {e}")

# Para ejecutar el modo interactivo, descomenta la siguiente línea:
# interactive_mode()

## 11. Ejemplo Completo: Análisis de Datos de Ventas

Vamos a demostrar todas las capacidades con un análisis completo.

In [ ]:
# Crear datos de ventas más completos
ventas_detalladas = """
DATOS DE VENTAS DETALLADOS - 2024

Ventas por mes y producto:

Enero:
- Producto A: 60 unidades a $250 = $15,000
- Producto B: 50 unidades a $300 = $15,000
- Producto C: 40 unidades a $375 = $15,000

Febrero:
- Producto A: 70 unidades a $250 = $17,500
- Producto B: 58 unidades a $300 = $17,400
- Producto C: 45 unidades a $375 = $16,875

Marzo:
- Producto A: 85 unidades a $250 = $21,250
- Producto B: 65 unidades a $300 = $19,500
- Producto C: 53 unidades a $375 = $19,875

Abril:
- Producto A: 95 unidades a $250 = $23,750
- Producto B: 72 unidades a $300 = $21,600
- Producto C: 58 unidades a $375 = $21,750

Costos de marketing mensuales:
- Enero: $5,000
- Febrero: $6,500
- Marzo: $8,000
- Abril: $9,500

ROI objetivo: 400%
"""

rag.add_document(ventas_detalladas, {"type": "detailed_sales", "year": 2024})
print("✅ Datos de ventas detallados agregados")

In [ ]:
# Análisis completo
response = agent.chat("""
Por favor realiza un análisis completo de los datos de ventas:

1. Busca en los documentos los datos de ventas mensuales totales
2. Calcula el crecimiento porcentual mes a mes
3. Calcula el ROI del marketing (Ventas/Costos marketing)
4. Crea un gráfico de línea mostrando la evolución de ventas mensuales
5. Realiza un análisis estadístico descriptivo de las ventas
6. Analiza la correlación entre costos de marketing y ventas

Presenta los resultados de forma estructurada y clara.
""")

## 12. Conclusión

Este notebook proporciona un sistema RAG avanzado con:

✅ **Gestión de documentos** - ChromaDB para búsqueda semántica

✅ **Cálculos matemáticos** - Calculadora con numpy y scipy

✅ **Ejecución de código** - Python ejecutable para análisis complejos

✅ **Visualizaciones** - Matplotlib, Seaborn, Plotly

✅ **Análisis estadístico** - Descriptivo, correlación, regresión, tests

✅ **Agente inteligente** - Function calling con Claude para orquestar herramientas

### Próximos pasos:

1. Cargar tus propios documentos con `rag.add_file(path)` o `rag.add_document(text)`
2. Hacer preguntas al agente sobre tus datos
3. Solicitar cálculos, análisis y visualizaciones
4. Extender con más herramientas según tus necesidades

### Ejemplos de preguntas que puedes hacer:

- "¿Cuáles son las ventas totales según los documentos?"
- "Calcula el promedio y desviación estándar de las ventas mensuales"
- "Crea un gráfico mostrando la tendencia de ventas"
- "Analiza la correlación entre marketing y ventas"
- "Ejecuta un código que simule 1000 ventas aleatorias y calcule estadísticas"
- "¿Cuál es el producto más rentable según los datos?"